In [30]:
import importlib
import inspect
import pkgutil
import re

import Products.zms as mypackage


def is_zope_exposed(owner, name, obj):
    """
    Detect common Zope publication/security patterns.
    """

    # foo__roles__ = None
    if hasattr(owner, f"{name}__roles__"):
        return True

    # Often added by Zope security machinery
    if hasattr(obj, "__roles__"):
        return True

    # Custom @zpublish decorator
    if hasattr(obj, "__zpublish__"):
        return True

    return False


def get_security_declared_methods(cls):
    """
    Return method names referenced by
    security.declarePublic(...)
    security.declareProtected(...)
    """

    try:
        source = inspect.getsource(cls)
    except Exception:
        return set()

    methods = set()

    methods.update(
        re.findall(
            r"declarePublic\s*\(\s*['\"]([^'\"]+)['\"]",
            source,
        )
    )

    methods.update(
        re.findall(
            r"declareProtected\s*\([^,]+,\s*['\"]([^'\"]+)['\"]",
            source,
        )
    )

    return methods


def is_public_documented(owner, name, obj):
    """
    True if member should appear in output.
    """

    if not inspect.getdoc(obj):
        return False

    if name.startswith("_"):
        return False

    if name.startswith("manage"):
        return False

    if is_zope_exposed(owner, name, obj):
        return False

    return True


def get_documented_members(module):
    """
    Yield all documented members that are not
    Zope-published APIs.
    """

    # ------------------------------------------------------------------
    # Module-level functions
    # ------------------------------------------------------------------
    for name, obj in inspect.getmembers(module, inspect.isfunction):

        if is_public_documented(module, name, obj):
            yield f"        '{name}',"

    # ------------------------------------------------------------------
    # Class methods
    # ------------------------------------------------------------------
    for cls_name, cls in inspect.getmembers(module, inspect.isclass):

        # Skip imported classes
        if cls.__module__ != module.__name__:
            continue

        security_methods = get_security_declared_methods(cls)

        for name, obj in cls.__dict__.items():

            # Take only methods defined directly in this class
            if not inspect.isfunction(obj):
                continue

            if name in security_methods:
                continue

            if is_public_documented(cls, name, obj):
                # yield f"{cls_name}.{name}"
                yield f"        '{name}',"


# ######################################################################
# MAIN
# ######################################################################

for info in pkgutil.walk_packages(
        mypackage.__path__,
        mypackage.__name__ + '.'):

    try:
        module = importlib.import_module(info.name)
    except ImportError:
        continue

    members = list(get_documented_members(module))

    if not members:
        continue

    print(f"    '{str(module.__name__).split('.')[-1]}': (")

    for member in sorted(members):
        print(f"{member}")
    print("    ),")

    'ZMSCharformatManager': (
        'delCharformat',
        'getCharFormats',
        'importCharformatXml',
        'moveCharformat',
        'setCharformat',
    ),
    'ZMSFilterManager': (
        'delFilter',
        'delFilterProcess',
        'delProcess',
        'exportXml',
        'getFilter',
        'getFilterIds',
        'getFilterProcesses',
        'getProcess',
        'getProcessIds',
        'importXml',
        'moveFilterProcess',
        'provideRepository',
        'setFilter',
        'setFilterProcess',
        'setProcess',
        'updateRepository',
    ),
    'ZMSFormatProviderAcquired': (
        'getCharFormats',
        'getTextFormat',
        'getTextFormatDefault',
        'getTextFormats',
    ),
    'ZMSItem': (
        'breadcrumbs_obj_path',
        'display_icon',
        'f_standard_html_request',
        'getTitlealt',
        'get_readme_path',
        'zmi_body_class',
        'zmi_body_content',
        'zmi_manage_menu',
        'zmi_pa